# Telco Customer Churn - Exploratory Data Analysis
## Business Question: Why are customers churning and what predicts retention?

In [1]:
import pandas as pd
import sqlite3

# Load raw CSV for inspection
df_raw = pd.read_csv('../data/Telco-Customer-Churn.csv')
df_raw.columns = df_raw.columns.str.lower().str.replace(' ', '_')

# Connect to database
conn = sqlite3.connect('../telco_churn.db')
print("Connected successfully")

Connected successfully


## Section 1: Data Inspection
Before cleaning anything, we inspect the raw data to understand its structure, 
data types, null values, and unique values per column.

In [2]:
# 1. Shape
print(f"Shape: {df_raw.shape}")
print(f"Shape: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
print(f"Columns: {list(df_raw.columns)}")

Shape: (7043, 21)
Shape: 7043, Columns: 21
Columns: ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


In [3]:
# 2. Column info - dtype, nulls, unique count
print("=" * 60)
print("COLUMN INFO")
print("=" * 60)

for col in df_raw.columns:
    dtype = df_raw[col].dtype
    nulls = df_raw[col].isna().sum()
    unique = df_raw[col].nunique()
    print(f"{col:<20} dtype: {str(dtype):<10} nulls: {str(nulls):<10} unique: {str(unique):<10}")

COLUMN INFO
customerid           dtype: str        nulls: 0          unique: 7043      
gender               dtype: str        nulls: 0          unique: 2         
seniorcitizen        dtype: int64      nulls: 0          unique: 2         
partner              dtype: str        nulls: 0          unique: 2         
dependents           dtype: str        nulls: 0          unique: 2         
tenure               dtype: int64      nulls: 0          unique: 73        
phoneservice         dtype: str        nulls: 0          unique: 2         
multiplelines        dtype: str        nulls: 0          unique: 3         
internetservice      dtype: str        nulls: 0          unique: 3         
onlinesecurity       dtype: str        nulls: 0          unique: 3         
onlinebackup         dtype: str        nulls: 0          unique: 3         
deviceprotection     dtype: str        nulls: 0          unique: 3         
techsupport          dtype: str        nulls: 0          unique: 3         


In [4]:
print("=" * 55)
print("NUMERIC SUMMARY")
print("=" * 55)
print(df_raw.describe())

NUMERIC SUMMARY
       seniorcitizen       tenure  monthlycharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


*** Realized that totalcharges is currently a str when it should be a float ***

In [5]:
# Check what's causing totalcharges to be a string
print("Sample totalcharges values:")
print(df_raw['totalcharges'].head(20).tolist())

print(f"\nBlank string count: {(df_raw['totalcharges'] == ' ').sum()}")
print(f"Empty string count: {(df_raw['totalcharges'] == '').sum()}")

Sample totalcharges values:
['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5', '1949.4', '301.9', '3046.05', '3487.95', '587.45', '326.8', '5681.1', '5036.3', '2686.05', '7895.15', '1022.95', '7382.25', '528.35', '1862.9']

Blank string count: 11
Empty string count: 0


## Section 2: Data Cleaning

### Issues identified in Section 1:
1. `totalcharges` is stored as string, it needs to be float64
2. 11 rows have blank strings in `totalcharges`, this need to be dropped

### Cleaning decisions:
- Convert `totalcharges` to numeric using `pd.to_numeric(errors='coerce')`
- Drop the 11 null rows — they represent 0.15% of data, negligible loss
- Work on a copy `df_clean` — never modify raw data



In [6]:
# Create clean copy - never modify raw data
df_clean = df_raw.copy()

# Fix 1: Convert totalcharges to float
df_clean['totalcharges'] = pd.to_numeric(df_clean['totalcharges'], errors='coerce')

# Fix 2: Drop rows where totalcharges is null (the 11 blank strings)
df_clean = df_clean.dropna(subset=['totalcharges'])

# Verify cleaning worked
print("=== CLEANING RESULTS ===")
print(f"Original rows: {len(df_raw)}")
print(f"Clean rows: {len(df_clean)}")
print(f"Rows dropped: {len(df_raw) - len(df_clean)}")
print(f"\ntotalcharges dtype: {df_clean['totalcharges'].dtype}")
print(f"totalcharges nulls: {df_clean['totalcharges'].isna().sum()}")
print(f"\nSample cleaned values: {df_clean['totalcharges'].head(5).tolist()}")

=== CLEANING RESULTS ===
Original rows: 7043
Clean rows: 7032
Rows dropped: 11

totalcharges dtype: float64
totalcharges nulls: 0

Sample cleaned values: [29.85, 1889.5, 108.15, 1840.75, 151.65]


**Checking for Class Imbalance in churn column. If 90% of data is "No Churn," model will struggle to learn what a "Churner" looks like.**
* Note: 73/27 split - moderate imbalance
* For this analysis we focus on SQL-based segmentation and A/B testing
* rather than predictive modeling. Class weighting would be applied
* if building a production ML model.

In [7]:
# Check the distribution of the target variable
print(df_clean['churn'].value_counts(normalize=True))

churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64


In [8]:
# Save cleaned dataset for use in subsequent notebooks
df_clean.to_csv('../data/telco_churn_clean.csv', index=False)

# Also update the SQLite database with clean data
conn_write = sqlite3.connect('../telco_churn.db')
df_clean.to_sql('customers', conn_write, if_exists='replace', index=False)
conn_write.close()

print(f"Clean data saved.")
print(f"Final shape: {df_clean.shape}")
print(f"Churn distribution:\n{df_clean['churn'].value_counts()}")

Clean data saved.
Final shape: (7032, 21)
Churn distribution:
churn
No     5163
Yes    1869
Name: count, dtype: int64
